# TTM Small4M — A/C one-packet T4 owner Colab

**Tình trạng khi xuất bản:** SOURCE_STAGED / NOT_EXECUTED. Notebook không có quyền tự chạy trong Colab khi chưa được chủ dự án mở và cấp phiên T4.

- Chỉ A/C, tuyệt đối không B/D hoặc TEST, Q1/Q2, FRESH12, CF1.
- 128 bước/arm, 4.096 valid labels/bước; nguồn P02, pack, checkpoint và runner đã khóa.
- Tải các dữ liệu riêng tư theo Google Drive ID đã xác minh, kiểm SHA-256 trước khi chạy.
- Chuẩn bị T4 FP16 zero-step smoke, mô hình thật P02 3,749,376 tham số, khôi phục trong kiểm thử synthetic; đây **không** phải science training.
- **Khóa run science không tự sinh và không tự ký.** Nếu thiếu quyền owner, người chấm độc lập và owner lock hợp lệ, trạng thái là HOLD.
- Không tái chạy Windows native V2 đã PASS.
- Checkpoint và mô hình huấn luyện sẽ chỉ được ghi vào thư mục Drive riêng của owner. Không push raw data hoặc key lên GitHub.
- FINAL24 phải được chấm mù và khóa điểm trước giải mù. Notebook này không tự nhận vai trò người chấm độc lập.

Mở notebook bằng **Open in Colab → GitHub**, chọn runtime GPU **NVIDIA T4**, sau đó Run all đúng một lần. Các kiểm tra đều fail-closed.
Sau khi huấn luyện đủ hai arm, Cell 5 chỉ xuất 48 đầu ra **owner-only** nếu có BPE manifest riêng tư đúng SHA; biên bản này KHÔNG chấm điểm hoặc giải mù.
Lưu ý phạm vi: kiểm tra khôi phục T4 bằng dữ liệu tổng hợp trên mô hình P02 thật chỉ đạt **PASS_SCOPED**; riêng cổng `PASS_ACTUAL_AC_RESUME_PARITY` phải do biên bản độc lập xác nhận, không tự nâng từ kiểm tra tổng hợp.
Tệp TRAIN/VAL và checkpoint được đọc theo **đường dẫn thư mục Drive của owner**, không công bố Google Drive file ID; mỗi tệp được kiểm SHA-256 trước khi sao chép vào môi trường Colab tạm.
**V4 FP16 backoff hotfix:** kiểm tra synthetic thực sự đếm optimizer step, thử lại sau khi GradScaler hạ scale (tối đa 12 lần), không bước scheduler nếu optimizer skip. Ghi scale transitions vào receipt; **không đổi runner science nguyên bản**, không tự cấp PASS actual AC resume hoặc science GO.

**V5 12/11/2026:** Cell 2B runs 7 negative QA tests on exact V2 AST and A/C two-step full-P02 real-TRAIN T4 resume QA (**8 ephemeral QA optimizer updates; 0 science steps**). Cell science invokes V2 only after exact owner lock, rights, independent reviewer and separately adjudicated resume. No automatic science GO from these QA receipts.

**V6 diagnostic only:** Cell 2B dùng subprocess stdout/stderr capture, không che traceback bằng CalledProcessError; khi chương trình con thất bại, lưu log riêng trên owner Drive và HOLD. Không sửa script huấn luyện V2, checkpoint, nguồn data, các khóa hoặc bất kỳ bước science nào.

**V7 fix checkpoint AdamW alias:** gói script V2 được khóa theo git blob mới; clone checkpoint/optimizer với `deepcopy` trước mỗi arm và trước resume để A/C không chia sẻ `optimizer.state.step` của parent; bổ sung guard bất biến bước 1024 và kiểm thử trên TRAIN thật. Dùng `/content/ttm_ac_objective_only_v7` để không tự ghi đè source cache V6; lưu riêng biên nhận QA V7 trên owner Drive. Chưa mở science GO.


In [ ]:
# Cell 1 — import + fixed source/Drive pin readback; no optimizer step.
import os, sys, json, hashlib, io, pathlib, urllib.request, subprocess, shutil, datetime
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
import numpy as np
import torch

REPO="minhtri22/TokenToModel-Lab"
AC_PIN="6628401b08ea7682cf34aa5078546e1d18e4d2cc"
EVAL_PIN="837289fda82b41653a61b74e476f76a0b4c4137c"
WORK=Path("/content/ttm_ac_objective_only_v7")
WORK.mkdir(parents=True,exist_ok=True)
PRIVATE=WORK/"private"
PRIVATE.mkdir(exist_ok=True)
OWNER=Path("/content/drive/MyDrive/TTM/ttm-small-pilot/ttm_small4m_ac_objective_only_v1")
OWNER.mkdir(parents=True,exist_ok=True)
QA=OWNER/"qa_preexec"
QA.mkdir(parents=True,exist_ok=True)
if str(WORK) not in sys.path: sys.path.insert(0,str(WORK))

sources={
"ttm_ac_objective_runner_candidate.py":("tools/pilot_t4/prompt_conditioning_2x2/ttm_ac_objective_runner_candidate.py","ad7d88672c65c6778f5956e6fc7aa656b78b998f"),
"ttm_ac_resume_store.py":("tools/pilot_t4/prompt_conditioning_2x2/ttm_ac_resume_store.py","8b02db35854cf24e77246d81dba6a8b8fe2d746e"),
"ttm_2x2_colab_runner_candidate.py":("tools/pilot_t4/prompt_conditioning_2x2/ttm_2x2_colab_runner_candidate.py","6c87e388da4d4a2b94dffba655861a390dfad73c"),
"real_metadata_preflight.py":("tools/pilot_t4/prompt_conditioning_2x2/real_metadata_preflight.py","84d54dced18b19bea2bc062216e32301c3c32dc2"),
"p02_source_exact.py":("src/p02/transformer_core.py","9762343e06cf800b50574c2871ae1ed58537fb18")}
eval_sources={
"FROZEN_FINAL24_PROMPTS.json":("docs/research/TTM_SMALL4M_2X2_NEW_DEV12_BLIND_FINAL24_PROMPTS_FROZEN_20261110.json","f08f15c5049baf77771bed463a5119136566373b"),
"BLIND24_ENCRYPTED_PUBLIC.json":("artifacts/qa/ttm-small4m-prompt-conditioning/ac_preexec/TTM_SMALL4M_AC_BLIND24_ENCRYPTED_ESCROW_PUBLIC_20261111.json","ff30fb1cded4712215f5ca657f4b98b182bc362f")}
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)
    return h.hexdigest()
def gitblob(raw):
    return hashlib.sha1(("blob "+str(len(raw))+"\0").encode("ascii")+raw).hexdigest()
def raw_fetch(ref,path):
    url="https://raw.githubusercontent.com/"+REPO+"/"+ref+"/"+path
    with urllib.request.urlopen(url,timeout=120) as req: return req.read()
for filename,(src,expect) in {**sources,**eval_sources}.items():
    target=WORK/filename
    if not target.exists():
        raw=raw_fetch(AC_PIN if filename in sources else EVAL_PIN,src)
        assert gitblob(raw)==expect,("SOURCE_GIT_BLOB_MISMATCH",filename)
        with target.open("xb") as f:f.write(raw)
    assert gitblob(target.read_bytes())==expect,("LOCAL_SOURCE_PIN_DRIFT",filename)
for filename in sources:
    if filename.endswith(".py"):compile((WORK/filename).read_text("utf-8"),filename,"exec")
# FP16 V2 candidate and two QA programs are pinned separately from frozen V1.
FP16_V2_QA_PIN="b9b265bbfc104c22a93d4c102fc3c9b9b2aeec7b"
QA_V2_SOURCE="ttm_ac_objective_runner_fp16_backoff_v2_REVIEW_CANDIDATE.py"
QA_V2_SOURCE_CHECK="independent_ac_fp16_v2_ast_fake_backend_qa.py"
QA_V2_REAL_TRAIN_CHECK="ttm_ac_fp16_v2_actual_train_resume_parity_preexec.py"
qa_sources={
  QA_V2_SOURCE:("tools/pilot_t4/prompt_conditioning_2x2/"+QA_V2_SOURCE,"f8b74f8439d70504028cdb94a5b01cb058a6107c"),
  QA_V2_SOURCE_CHECK:("tools/pilot_t4/prompt_conditioning_2x2/"+QA_V2_SOURCE_CHECK,"c027ab9228fd88024d9e27db6e6f178e63b63aef"),
  QA_V2_REAL_TRAIN_CHECK:("tools/pilot_t4/prompt_conditioning_2x2/"+QA_V2_REAL_TRAIN_CHECK,"1ca7092847c28d525203e1045394ce9caf4a86dc")}
for filename,(github_path,blob) in qa_sources.items():
    target=WORK/filename
    if not target.exists():
        raw=raw_fetch(FP16_V2_QA_PIN,github_path)
        assert gitblob(raw)==blob,("V2_GIT_BLOB_INVALID",filename)
        with target.open("xb") as out:out.write(raw)
    assert gitblob(target.read_bytes())==blob,("V2_SOURCE_LOCAL_DRIFT",filename)
    compile(target.read_text("utf-8"),filename,"exec")


# Only mounted owner-private Drive FOLDER paths. Never publish private Drive IDs.
T4ROOT=Path("/content/drive/MyDrive/TTM/ttm-small-pilot")
drive_inputs={
"T4_PRIVATE_TRAIN_VAL_WINDOWS.npz":(T4ROOT/"t4_private_pack_20261108_v1"/"T4_PRIVATE_TRAIN_VAL_WINDOWS.npz","184f68b4c0c800ef3e98362c2e20e1cd35baa269e926489a475664526ecd9913"),
"T4_PRIVATE_PACK_MANIFEST.json":(T4ROOT/"t4_private_pack_20261108_v1"/"T4_PRIVATE_PACK_MANIFEST.json","ef2e6e67d49b1d5a921b1051df329f15574f1c10cece0ce49468acb53322d650"),
"checkpoint_step1024.pt":(T4ROOT/"t4_science_4m_1024_a2_20261108_v1"/"checkpoints"/"step_1024"/"checkpoint.pt","cb985751795c8328ad65acf7ca4c1033ce35182c1bfbdb8c38550de629cb271b")}
for filename,(source_path,expected_sha) in drive_inputs.items():
    assert source_path.is_file() and not source_path.is_symlink(),("OWNER_DRIVE_SOURCE_MISSING",filename,str(source_path))
    assert sha256(source_path)==expected_sha,("OWNER_DRIVE_SOURCE_SHA_MISMATCH",filename)
    destination=PRIVATE/filename
    if not destination.exists():
        with source_path.open("rb") as src,destination.open("xb") as out:
            shutil.copyfileobj(src,out,1024*1024)
            out.flush();os.fsync(out.fileno())
    assert sha256(destination)==expected_sha,("PRIVATE_LOCAL_SHA_MISMATCH",filename)
print("AC_PINNED_SOURCES_AND_INPUTS=PASS")
print("PYTORCH_RUNTIME=",torch.__version__,"CUDA_AVAILABLE=",torch.cuda.is_available())
print("OWNER_DRIVE_ROOT=",OWNER)


In [ ]:
# Cell 2 — Full actual P02/T4 FP16 smoke and interrupted serialize/reload parity, SYNTHETIC INPUTS ONLY.
import copy, random
import ttm_ac_objective_runner_candidate as ac
import ttm_2x2_colab_runner_candidate as base
import ttm_ac_resume_store as store

assert torch.cuda.is_available() and torch.cuda.device_count()==1,"ONE_T4_GPU_REQUIRED"
assert "T4" in torch.cuda.get_device_name(0),"NVIDIA_T4_REQUIRED"
torch.backends.cudnn.benchmark=False
dev=torch.device("cuda")
p02=base.load_model_source(WORK/"p02_source_exact.py")
cp=base.check_input(PRIVATE/"T4_PRIVATE_TRAIN_VAL_WINDOWS.npz",
                    PRIVATE/"T4_PRIVATE_PACK_MANIFEST.json",
                    PRIVATE/"checkpoint_step1024.pt")
assert cp["step"]==1024,"BAD_SOURCE_PARENT_STEP"
def write_once_json(path,obj):
    with path.open("x",encoding="utf-8") as f:
        json.dump(obj,f,sort_keys=True,indent=2)
        f.write("\n"); f.flush();os.fsync(f.fileno())
def receipt_existing(path,expected_status):
    if not path.exists():return False
    j=json.loads(path.read_text("utf-8"))
    assert j.get("status")==expected_status and j.get("checkpoint_sha256")==base.CHECKPOINT_SHA,"QA_RECEIPT_PIN_DRIFT"
    return True

smoke_file=QA/"T4_FP16_ZERO_STEP.json"
resume_file=QA/"AC_REAL_P02_T4_FP16_RESUME_PARITY.json"
if not receipt_existing(smoke_file,"PASS_T4_FP16_ZERO_SCIENCE_STEP"):
    model,opt,sch,sampler,scaler=base.clone_parent(p02,cp,dev)
    model.eval()
    xx=torch.arange(256,dtype=torch.long,device=dev).view(1,256)%2048
    with torch.no_grad(),torch.autocast("cuda",dtype=torch.float16):
        logits,_=model(xx)
    assert logits.shape==(1,256,2048) and torch.isfinite(logits).all().item(),"T4_SMOKE_NONFINITE"
    assert len(model.state_dict())>0 and sum(p.numel() for p in model.parameters())==3749376
    write_once_json(smoke_file,{"schema":"ttm.small4m.ac.t4.real_parent.fp16.zero_step.v1",
        "status":"PASS_T4_FP16_ZERO_SCIENCE_STEP","checkpoint_sha256":base.CHECKPOINT_SHA,
        "model_sha256":base.P02_SHA,"torch_version":torch.__version__,
        "gpu_name":torch.cuda.get_device_name(0),
        "model_parameters":3749376,"training_steps":0,"train_val_tokens_scored":0,
        "model_forward":"FP16_SYNTHETIC_IDS_ONLY","test_access":False})
    del model,opt,sch,sampler,scaler
    torch.cuda.empty_cache()

def full_step(parts,x,y,*,max_scale_backoffs=12):
    """Synthetic-input FP16 QA ONLY: retry GradScaler overflow until one true AdamW update.

    A skipped optimizer step must NOT advance the scheduler. This is not a
    science-runner policy change and cannot grant PASS_ACTUAL_AC_RESUME_PARITY.
    """
    model,opt,sch,sampler,scaler=parts
    assert opt.state, "FROZEN_ADAMW_STATE_MISSING"
    state_key=next(iter(opt.state))
    observations=[]
    for attempt in range(max_scale_backoffs+1):
        model.train()
        opt.zero_grad(set_to_none=True)
        before=float(opt.state[state_key]["step"].item())
        before_lr=float(opt.param_groups[0]["lr"])
        before_epoch=int(sch.last_epoch)
        for loss in base.loss_step(model,x,y,"A",dev,autocast=True):
            scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        old_scale=float(scaler.get_scale())
        scaler.step(opt)
        scaler.update()
        new_scale=float(scaler.get_scale())
        after=float(opt.state[state_key]["step"].item())
        observations.append({"attempt":attempt+1,
                             "scale_before":old_scale,
                             "scale_after":new_scale,
                             "optimizer_step_before":before,
                             "optimizer_step_after":after})
        if after==before+1.0:
            # Only a REAL optimizer update is allowed to move LR scheduler.
            sch.step()
            assert sch.last_epoch==before_epoch+1, "QA_SCHEDULER_STEP_DRIFT"
            return {"optimizer_updates":1,
                    "scaler_skipped_attempts":attempt,
                    "original_parent_scale_skip_observed":any(
                        item["scale_after"]<item["scale_before"] for item in observations),
                    "attempts":observations}
        assert after==before, "QA_OPTIMIZER_STEP_STATE_DRIFT"
        assert new_scale<old_scale, "QA_SKIPPED_STEP_WITHOUT_GRADSCALER_BACKOFF"
        assert sch.last_epoch==before_epoch, "QA_SCHEDULER_MOVED_ON_SKIPPED_STEP"
        assert float(opt.param_groups[0]["lr"])==before_lr, "QA_LR_MOVED_ON_SKIPPED_STEP"
    raise RuntimeError("HOLD_FP16_SYNTHETIC_QA_CANNOT_COMPLETE_OPTIMIZER_UPDATE_AFTER_BACKOFFS")

def same_models_and_state(p,q):
    m1,o1,s1,g1,z1=p
    m2,o2,s2,g2,z2=q
    a=m1.state_dict(); b=m2.state_dict()
    if a.keys()!=b.keys() or not all(torch.equal(a[k],b[k]) for k in a):return False
    ao=o1.state_dict();bo=o2.state_dict()
    if ao["param_groups"]!=bo["param_groups"] or ao["state"].keys()!=bo["state"].keys():return False
    for k,v in ao["state"].items():
        for attr,val in v.items():
            other=bo["state"][k][attr]
            if torch.is_tensor(val):
                if not torch.equal(val,other):return False
            elif val!=other:return False
    return s1.state_dict()==s2.state_dict() and z1.state_dict()==z2.state_dict() and torch.equal(g1.get_state(),g2.get_state())

if not receipt_existing(resume_file,"PASS_REAL_P02_T4_SYNTHETIC_RELOAD_PARITY_SCOPED"):
    # Real 3.75M P02 and locked step1024 optimizer/scaler/seed, but NO real TRAIN examples.
    g=np.random.default_rng(20261111)
    x=g.integers(0,2048,size=(16,256),dtype=np.int64)
    y=np.roll(x,-1,axis=1)
    assert int((y!=-100).sum())==4096
    parts=base.clone_parent(p02,cp,dev)
    qa_step1=full_step(parts,x,y)
    payload=ac.state_dump(*parts,1025,[],"A")
    snap=QA/"AC_P02_T4_SYNTHETIC_QA_AFTER_FIRST_STEP.pt"
    assert not snap.exists(),"QA_SNAPSHOT_ALREADY_EXISTS_MANUAL_REVIEW_REQUIRED"
    with snap.open("xb") as fp:
        torch.save(payload,fp);fp.flush();os.fsync(fp.fileno())
    sha=sha256(snap)
    qa_step2=full_step(parts,x,y) # uninterrupted second full-model optimizer update
    loaded=torch.load(snap,map_location="cpu",weights_only=True)
    restored=ac.restore_arm(p02,cp,loaded,dev)
    qa_resumed=full_step(restored,x,y) # resume at the first synthetic checkpoint
    assert qa_step2["attempts"]==qa_resumed["attempts"],"QA_RESUMED_SCALE_TRANSITIONS_DRIFT"
    assert same_models_and_state(parts,restored),"REAL_P02_T4_RESUME_PARITY_FAILED"
    write_once_json(resume_file,{"schema":"ttm.small4m.ac.t4.p02.full_parent.synthetic_input_resume.v1",
        "status":"PASS_REAL_P02_T4_SYNTHETIC_RELOAD_PARITY_SCOPED","checkpoint_sha256":base.CHECKPOINT_SHA,
        "model_sha256":base.P02_SHA,"torch_version":torch.__version__,
        "gpu_name":torch.cuda.get_device_name(0),
        "scope":"REAL_P02_3_749_376_PARAMETER_REAL_PARENT_ADAMW_SCALER_SCHEDULER_ON_T4_WITH_SYNTHETIC_IDS",
        "restart_method":"TORCH_SAVE_WEIGHTS_ONLY_LOAD_THEN_RESTORE_ARM","snapshot_sha256":sha,
        "comparison":"BIT_EXACT_MODEL_OPTIMIZER_SCALER_SCHEDULER_SAMPLER",
        "test_access":False,"private_train_labels_scored":0,"science_train_steps":0,
        "synthetic_qa_optimizer_updates":3,
        "qa_step1":qa_step1,"qa_step2":qa_step2,
        "qa_resumed":qa_resumed,
        "source_science_runner_overflow_policy":"UNCHANGED_FAIL_ON_SKIP",
        "actual_AC_resume_gate":"NOT_GRANTED_BY_THIS_SYNTHETIC_QA"})
    del parts,restored
    torch.cuda.empty_cache()

# Frozen 24-prompt source and encrypted escrow are checked, NOT read as prior TEST.
evalfile=QA/"FINAL24_EVAL_SOURCE_FREEZE.json"
if not receipt_existing(evalfile,"PASS_24_PROMPT_BLIND_FREEZE"):
    prompts=json.loads((WORK/"FROZEN_FINAL24_PROMPTS.json").read_text("utf-8"))
    final=[q for q in prompts["prompts"] if q["split"]=="BLIND_FINAL"]
    assert len(final)==24 and [q["id"] for q in final]==["Q"+str(i).zfill(2) for i in range(1,25)]
    for q in final:
        assert hashlib.sha256(q["prompt"].encode("utf-8")).hexdigest()==q["utf8_sha256"]
    escrow=WORK/"BLIND24_ENCRYPTED_PUBLIC.json"
    assert sha256(escrow)=="519f3185d6202052ce1e5c98a7e7fa58a813d87ce9d9e2d173eb25f6a05f9f1c"
    write_once_json(evalfile,{"status":"PASS_24_PROMPT_BLIND_FREEZE",
        "checkpoint_sha256":base.CHECKPOINT_SHA,
        "prompts_git_blob":"f08f15c5049baf77771bed463a5119136566373b",
        "encrypted_escrow_sha256":sha256(escrow),"final_count":24,
        "scoring":"NOT_EXECUTED","test_access":False})
if not (QA/"COLAB_ENVIRONMENT.json").exists():
    write_once_json(QA/"COLAB_ENVIRONMENT.json",{
        "python":sys.version.split()[0],"torch":torch.__version__,
        "gpu":torch.cuda.get_device_name(0),"science_optimizer_steps":0,
        "sources":{"ac_pin":AC_PIN,"p02_blob":"9762343e06cf800b50574c2871ae1ed58537fb18"},
        "input_sha256":{k:v[1] for k,v in drive_inputs.items()}})
if not (QA/"COLAB_PIP_FREEZE.txt").exists():
    with (QA/"COLAB_PIP_FREEZE.txt").open("x",encoding="utf-8") as out:
        out.write(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
print("T4_FP16_ZERO_STEP_AND_P02_REAL_PARENT_SYNTHETIC_RESUME=PASS_SCOPED")
print("AC_SCIENCE_STEPS=0")


In [ ]:
# Cell 2B — V6: child stderr/traceback diagnostics for actual P02 REAL-TRAIN T4 resume QA.
# Uses 8 EPHEMERAL optimizer updates (4 per A/C) strictly as pre-science QA,
# never as any part of 128-step/arm science. Never access TEST/Q1Q2/OOD.
QA_V2_FILE=QA/"AC_FP16_V2_PARENT_ISOLATED_AST_7_OF_7_V7.json"
if not QA_V2_FILE.exists():
    subprocess.check_call([sys.executable,str(WORK/QA_V2_SOURCE_CHECK),
                           "--candidate",str(WORK/QA_V2_SOURCE),
                           "--out",str(QA_V2_FILE)])
v2_static=json.loads(QA_V2_FILE.read_text("utf-8"))
assert v2_static["status"]=="PASS_SCOPED_SOURCE_AST_FAKE_BACKEND_7_OF_7"
assert v2_static["candidate_git_blob"]=="f8b74f8439d70504028cdb94a5b01cb058a6107c"
assert v2_static["ac_science_training"] is False
print("AC_FP16_V2_SOURCE_QA=PASS_SCOPED_7_OF_7")

V2_REAL_T4=QA/"AC_FP16_V2_PARENT_ISOLATED_REAL_P02_REAL_TRAIN_TWO_STEP_PARITY_V7.json"
if not V2_REAL_T4.exists():
    assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0)
    cmd=[sys.executable,"-u","-X","faulthandler",str(WORK/QA_V2_REAL_TRAIN_CHECK),
         "--candidate",str(WORK/QA_V2_SOURCE),
         "--base",str(WORK/"ttm_2x2_colab_runner_candidate.py"),
         "--store",str(WORK/"ttm_ac_resume_store.py"),
         "--p02",str(WORK/"p02_source_exact.py"),
         "--pack",str(PRIVATE/"T4_PRIVATE_TRAIN_VAL_WINDOWS.npz"),
         "--manifest",str(PRIVATE/"T4_PRIVATE_PACK_MANIFEST.json"),
         "--checkpoint",str(PRIVATE/"checkpoint_step1024.pt"),
         "--out",str(V2_REAL_T4)]
    # Capture the CHILD traceback, not just subprocess.CalledProcessError.
    # Failure receipt remains on OWNER-PRIVATE Drive; never copy traceback
    # (including private paths) to GitHub, reviewer packet or public output.
    proc=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE,
                        text=True,errors="replace",check=False)
    if proc.stdout:
        print("AC_V2_CHILD_STDOUT_TAIL=\\n"+proc.stdout[-6000:])
    if proc.returncode:
        print("AC_V2_CHILD_EXIT_CODE=",proc.returncode)
        print("AC_V2_CHILD_STDERR_TAIL=\\n"+(proc.stderr[-12000:] or "<EMPTY_CHILD_STDERR>"))
        from datetime import datetime,timezone
        import uuid
        when=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        diag=QA/("AC_FP16_V2_T4_CHILD_ERROR_"+when+"_"+uuid.uuid4().hex[:8]+".json")
        details={"schema":"ttm.small4m.ac.v2.preexec.child_failure_diagnostic.v1",
                 "status":"HOLD_CHILD_EXIT_NONZERO",
                 "exit_code":proc.returncode,
                 "stdout_tail":proc.stdout[-12000:],
                 "stderr_tail":proc.stderr[-25000:],
                 "candidate_git_blob":"f8b74f8439d70504028cdb94a5b01cb058a6107c",
                 "child_script_git_blob":"1ca7092847c28d525203e1045394ce9caf4a86dc",
                 "python_executable":sys.executable,
                 "torch_version":torch.__version__,
                 "gpu_name":torch.cuda.get_device_name(0),
                 "science_optimizer_steps":0,
                 "test_q1q2_ood_access":False,
                 "not_independent_science_QA":True}
        write_once_json(diag,details)
        print("AC_V2_OWNER_PRIVATE_DIAGNOSTIC=",diag.name)
        print("AC_SCIENCE=HOLD_NO_GPU_SCIENCE_EXECUTION")
        raise RuntimeError("AC_V2_REAL_T4_PREEXEC_CHILD_FAILED; read child STDERR above, not CalledProcessError")
    if proc.stderr:
        print("AC_V2_CHILD_STDERR_NONFATAL_TAIL=\\n"+proc.stderr[-1500:])
v2_real=json.loads(V2_REAL_T4.read_text("utf-8"))
assert v2_real["status"]=="PASS_REAL_P02_T4_REAL_TRAIN_TWO_STEP_RESUME_PARITY_SCOPED"
assert v2_real["candidate_git_blob"]=="f8b74f8439d70504028cdb94a5b01cb058a6107c"
assert v2_real["qa_real_train_optimizer_updates"]==8
assert v2_real["science_optimizer_steps"]==0
assert all(a["exact_tensor_optimizer_scheduler_scaler_rng_parity"] for a in v2_real["arm_results"])
assert all(a["frozen_step1024_parent_optimizer_immutable"] and a["parent_step_tensor_storage_unaliased"] for a in v2_real["arm_results"])
print("AC_FP16_V2_REAL_T4_QA=PASS_SCOPED_A_C_2STEP_RESTART")
print("ACTUAL_AC_RESUME_SCIENCE_GATE=PENDING_INDEPENDENT_ADJUDICATION")
print("AC_SCIENCE_STEPS=0")


In [ ]:
# Cell 3 — One integrated OWNER SCIENCE gate. NO generated or forged owner/reviewer attestations.
import argparse
RIGHTS=OWNER/"RIGHTS_OWNER_SIGNED.json"
REVIEWER=OWNER/"REVIEWER_INDEPENDENCE_SIGNED.json"
LOCK=OWNER/"OWNER_AC_SCIENCE_LOCK.json"
NEEDS=[str(p) for p in (RIGHTS,REVIEWER,LOCK) if not p.is_file()]
if NEEDS:
    print("AC_SCIENCE=NO_GO_PREEXEC_MISSING_OWNER_ATTESTATION_REVIEWER_OR_LOCK")
    print("MISSING_OWNER_LOCAL_FILES=",NEEDS)
    print("NO_NEW_TRAINING_OR_FINAL24_GENERATION")
else:
    rights=json.loads(RIGHTS.read_text("utf-8"))
    reviewer=json.loads(REVIEWER.read_text("utf-8"))
    assert rights.get("status")=="PASS_OWNER_ATTESTED" and rights.get("owner_identifier"),"OWNER_RIGHTS_SIGNOFF_INVALID"
    assert rights.get("private_pack_sha256")==base.PACK_SHA,"OWNER_RIGHTS_PACK_MISMATCH"
    assert reviewer.get("status")=="PASS_INDEPENDENT_REVIEWER_LOCKED" and reviewer.get("reviewer_identity"),"REVIEWER_ATTESTATION_INVALID"
    assert reviewer.get("score_freeze_before_unblind") is True,"REVIEWER_BLIND_FREEZE_NOT_ACCEPTED"
    out=OWNER/"TTM_SMALL4M_AC_OBJECTIVE_ONLY_V1_SCIENCE"
    resume_requested=out.exists()
    args=argparse.Namespace(
        mode="science",pack=str(PRIVATE/"T4_PRIVATE_TRAIN_VAL_WINDOWS.npz"),
        manifest=str(PRIVATE/"T4_PRIVATE_PACK_MANIFEST.json"),
        checkpoint=str(PRIVATE/"checkpoint_step1024.pt"),
        p02=str(WORK/"p02_source_exact.py"),output=str(out),
        resume=resume_requested,
        owner_lock=str(LOCK),owner_lock_sha=sha256(LOCK),
        rights_receipt=str(RIGHTS),reviewer_receipt=str(REVIEWER),
        eval_receipt=str(evalfile),
        resume_qa_receipt=str(OWNER/"ACTUAL_AC_RESUME_INDEPENDENT_ADJUDICATION.json"),
        t4_smoke_receipt=str(smoke_file))
    # Owner lock MUST pin the V2 science runner and amended FP16 policy.
    # Scoping two-step QA does NOT auto-grant independent actual resume gate.
    # Strict frozen runner validation checks source+lock SHA, rights/reviewer
    # receipt SHA, known 2 arms, exact train budget and GPU/torch version.
    import ttm_ac_objective_runner_fp16_backoff_v2_REVIEW_CANDIDATE as ac_science_v2
    assert v2_static["status"]=="PASS_SCOPED_SOURCE_AST_FAKE_BACKEND_7_OF_7"
    assert v2_real["status"]=="PASS_REAL_P02_T4_REAL_TRAIN_TWO_STEP_RESUME_PARITY_SCOPED"
    lock=ac_science_v2.lock_science(args)
    assert len(lock.get("arms",[]))==2 and lock["arms"]==["A","C"]
    print("AC_OWNER_ONE_SHOT_SCIENCE_LOCK=PASS")
    ac_science_v2.science(args,lock)
    print("AC_SCIENCE=TRAINED_BOTH_ARMS_UNADJUDICATED")


In [ ]:
# Cell 4 — post-science public-safe readback; NO unblinding and NO reviewer impersonation.
science_root=OWNER/"TTM_SMALL4M_AC_OBJECTIVE_ONLY_V1_SCIENCE"
status_file=OWNER/"AC_OWNER_SCIENCE_READBACK.json"
if not science_root.is_dir():
    print("AC_OWNER_FINAL_STATUS=HOLD_NO_SCIENCE")
else:
    finals=[]
    for arm in ("A","C"):
        f=science_root/("ARM_"+arm)/"FINAL.json"
        if f.is_file():
            rec=json.loads(f.read_text("utf-8"))
            assert rec.get("arm")==arm and rec.get("status")=="TRAINED_CHECKPOINT_UNADJUDICATED"
            assert rec.get("target_exposures")==524288 and rec.get("step")==1152
            finals.append({"arm":arm,"checkpoint_sha256":rec["ready_checkpoint_sha256"],
                           "val_nll":rec["val_report"]["val_nll"],
                           "target_exposures":524288})
    if len(finals)==2:
        doc={"schema":"ttm.small4m.ac.two_arm.owner_science_unadjudicated.v1",
             "status":"TRAINED_CHECKPOINTS_UNADJUDICATED",
             "arms":finals,
             "causal_C_minus_A":"NOT_ESTIMABLE_UNTIL_FROZEN_FINAL24_BLIND_SCORING",
             "test_access":False,"Q1Q2_access":False,"OOD_access":False}
        if not status_file.exists():write_once_json(status_file,doc)
        else:assert json.loads(status_file.read_text("utf-8"))==doc,"OWNER_READBACK_CHANGED"
        print("AC_SCIENCE_TRAINING=COMPLETE_UNADJUDICATED")
        print("FINAL24=REQUIRES_SEPARATE_INDEPENDENT_BLIND_REVIEW_AND_SCORE_SHA_FREEZE")
        print("AC_OWNER_REPORT=",status_file)
    else:
        print("AC_SCIENCE=PARTIAL_OR_HOLD, DO_NOT_CLAIM_C_MINUS_A")


In [ ]:
# Cell 5 — Prepare all 48 frozen FINAL24 continuations for owner-only blind-packet escrow.
# This stage requires the actual two completed arm checkpoints AND the owner's
# original private BPE manifest. NEVER score, unblind or distribute arm labels here.
BPE_MANIFEST=OWNER/"T1_BPE2048_REAL_TRAIN_MANIFEST.json"
GENERATIONS=OWNER/"OWNER_ONLY_AC_FINAL24_48_RAW_GENERATIONS.json"
TOK_SOURCE=WORK/"ttm_gpt2_bpe2048.py"
BPE_MANIFEST_SHA="cba18e60cee3d08a353ee6d4a451017289e45af7b4b1179c87171b9b78d7dfdc"
if not science_root.is_dir() or not all((science_root/("ARM_"+arm)/"FINAL.json").exists() for arm in ("A","C")):
    print("FINAL24_GENERATION=HOLD_UNADJUDICATED_TWO_ARM_SCIENCE_NOT_FINISHED")
elif not BPE_MANIFEST.is_file():
    print("FINAL24_GENERATION=HOLD_OWNER_PRIVATE_BPE_MANIFEST_NOT_ON_DRIVE")
    print("PRIVATE_FILE_NEEDED=",BPE_MANIFEST)
elif GENERATIONS.exists():
    print("FINAL24_GENERATION=ALREADY_EXISTS_NO_OVERWRITE")
else:
    assert sha256(BPE_MANIFEST)==BPE_MANIFEST_SHA,"TRAIN_ONLY_BPE_MANIFEST_SHA_DRIFT"
    if not TOK_SOURCE.exists():
        src="tools/pilot_t4/integrated/source_t2/ttm_gpt2_bpe2048.py"
        raw=raw_fetch(AC_PIN,src)
        assert gitblob(raw)=="fa5bd892941ba5aa6e6b2f0b2b0ae974efe297b6"
        with TOK_SOURCE.open("xb") as out:out.write(raw)
    assert gitblob(TOK_SOURCE.read_bytes())=="fa5bd892941ba5aa6e6b2f0b2b0ae974efe297b6"
    import importlib.util
    spec=importlib.util.spec_from_file_location("ttm_ac_bpe_frozen",TOK_SOURCE)
    bpe_mod=importlib.util.module_from_spec(spec)
    sys.modules[spec.name]=bpe_mod
    spec.loader.exec_module(bpe_mod)
    bpe=bpe_mod.BPE2048.from_manifest(json.loads(BPE_MANIFEST.read_text("utf-8")))
    prompts_doc=json.loads((WORK/"FROZEN_FINAL24_PROMPTS.json").read_text("utf-8"))
    frozen_final=[x for x in prompts_doc["prompts"] if x["split"]=="BLIND_FINAL"]
    assert len(frozen_final)==24
    raw_outputs=[]
    for arm in ("A","C"):
        arm_dir=science_root/("ARM_"+arm)
        # Verify the READY-last checkpoint SHA and owner/science source scope.
        last,state,ready=store.load_latest(arm_dir,ac_science_v2.RUN_ID,arm,
            sha256(WORK/QA_V2_SOURCE),
            base.CHECKPOINT_SHA,sha256(LOCK))
        assert last==1152 and ready and len(state["history"])==128
        model=p02.GPT(p02.P02Config(**cp["config"])).to(dev)
        model.load_state_dict(state["model"],strict=True)
        model.eval()
        for prompt in frozen_final:
            assert hashlib.sha256(prompt["prompt"].encode("utf-8")).hexdigest()==prompt["utf8_sha256"]
            prefix=bpe.encode(prompt["prompt"])
            assert len(prefix)==prompt["bpe_token_count"],"FROZEN_PROMPT_TOKEN_COUNT_MISMATCH"
            tokens=list(prefix); generated=[]
            for _ in range(64):
                xx=torch.tensor([tokens[-256:]],dtype=torch.long,device=dev)
                with torch.no_grad(),torch.autocast("cuda",dtype=torch.float16):
                    logits,_=model(xx)
                nxt=int(torch.argmax(logits[0,-1]).item())
                generated.append(nxt);tokens.append(nxt)
                if nxt==bpe_mod.EOS_ID:break
            text_out=bpe.decode(generated,allow_eos=True)
            raw_outputs.append({"prompt_id":prompt["id"],"arm":arm,
                "output":text_out,"output_utf8_sha256":hashlib.sha256(text_out.encode("utf-8")).hexdigest(),
                "new_bpe_tokens":len(generated),"checkpoint_sha256":ready["checkpoint_sha256"]})
        del model
        torch.cuda.empty_cache()
    assert len(raw_outputs)==48
    # Owner-private Arm A/C mapping. Reviewer must NOT receive this file.
    write_once_json(GENERATIONS,{"status":"UNADJUDICATED_OWNER_ONLY",
        "run_id":ac_science_v2.RUN_ID,"outputs":raw_outputs,
        "frozen_prompt_git_blob":"f08f15c5049baf77771bed463a5119136566373b",
        "decoder":"GREEDY_ARGMAX_MAX64","test_access":False})
    print("FINAL24_OWNER_ONLY_GENERATION=48_OUTPUTS_UNADJUDICATED")
    print("OWNER_ONLY_RAW_GENERATION_SHA256=",sha256(GENERATIONS))
    print("REVIEWER_PACKET=REQUIRES_OWNER_PRIVATE_AES_ESCROW_AND_BLIND_MAPPING")
